# Fine tuning
- *Rodado em*: https://colab.research.google.com/drive/1DjqOnKIXzIr9KiI-3bLIQwIbEntpjh2L#scrollTo=J4Kzqvw9f-MS

## Instalar Roboflow

In [2]:
!pip install roboflow # install roboflow

  Using cached roboflow-1.3.8-py3-none-any.whl.metadata (11 kB)
  Using cached idna-3.7-py3-none-any.whl.metadata (9.9 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached kiwisolver-1.5.0-cp314-cp314-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.1 kB)
  Using cached matplotlib-3.10.9-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (52 kB)
  Using cached numpy-2.4.4-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached opencv_python_headless-4.10.0.84-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (20 kB)
  Using cached pillow-12.2.0-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (8.8 kB)
  Using cached pi_heif-1.3.0-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.3 kB)
  Using cached pillow_avif_plugin-1.5.5-cp314-cp314-manylinux_2_28_x86_64.whl.metadata (2.2 kB)
  Using cached python_dotenv-1.2.2-py3-none-any.whl.metad

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="0ebDczdh4jeiu2qK9u0z")
project = rf.workspace("roboflow-universe-projects").project("license-plate-recognition-rxg4e")
version = project.version(13)
dataset = version.download("yolov8")


loading Roboflow workspace...
loading Roboflow project...


## Treino

In [ ]:
import json
import os
from pathlib import Path

results = model.train(
    data=os.path.join(dataset.location, "data.yaml"),
    epochs=1,
    imgsz=640,
    batch=16,
    workers=2,
    device=0
)

## Copiamos os pesos para um arquivo


In [ ]:
os.makedirs("saved_models", exist_ok=True)
shutil.copy("runs/detect/train/weights/best.pt", "runs/detect/train/weights/license_plate_best.pt")
shutil.copy("runs/detect/train/weights/last.pt", "runs/detect/train/weights/license_plate_last.pt")
if os.path.exists("runs/detect/train/weights/license_plate_best.pt"):
    print("Pesos salvos com sucesso!")

## Extraindo metricas do modelo

Esta etapa roda a validacao do melhor checkpoint e salva um resumo com as principais metricas de deteccao.

In [ ]:
best_weights = Path("runs/detect/train/weights/best.pt")
metrics_output = Path("saved_models/metrics_summary.json")

if not best_weights.exists():
    raise FileNotFoundError(f"Checkpoint nao encontrado: {best_weights}")

best_model = YOLO(str(best_weights))
val_metrics = best_model.val(
    data=os.path.join(dataset.location, "data.yaml"),
    split="val",
    imgsz=640,
    batch=16,
    device=0,
    plots=True,
)

box_metrics = val_metrics.box
metrics_summary = {
    "weights": str(best_weights),
    "dataset_yaml": os.path.join(dataset.location, "data.yaml"),
    "split": "val",
    "precision": float(box_metrics.p),
    "recall": float(box_metrics.r),
    "map50": float(box_metrics.map50),
    "map75": float(box_metrics.map75),
    "map50_95": float(box_metrics.map),
    "fitness": float(val_metrics.fitness),
    "speed_ms": {k: float(v) for k, v in val_metrics.speed.items()},
    "per_class_map50_95": {
        str(best_model.names[idx]): float(score)
        for idx, score in enumerate(box_metrics.maps)
    },
    "raw_results": {
        key: float(value)
        for key, value in val_metrics.results_dict.items()
    },
}

metrics_output.write_text(
    json.dumps(metrics_summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(json.dumps(metrics_summary, indent=2, ensure_ascii=False))
print(f"Metricas salvas em: {metrics_output}")